# Mega Net
Courtesy of ChatGPT

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class UltraAdvancedCNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Initial convolutional block
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        
        # Residual block 1
        self.res_block1 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64)
        )
        
        # Downsampling block
        self.downsample1 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )
        
        # Residual block 2
        self.res_block2 = nn.Sequential(
            nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128)
        )
        
        # Downsampling block
        self.downsample2 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU()
        )
        
        # Residual block 3
        self.res_block3 = nn.Sequential(
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256)
        )
        
        # Fully connected layers
        self.fc1 = nn.Sequential(
            nn.Linear(256 * 7 * 7, 1024),
            nn.ReLU(),
            nn.Dropout(0.5)
        )
        self.fc2 = nn.Sequential(
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.5)
        )
        self.fc3 = nn.Linear(512, 10)
    
    def forward(self, x):
        # Convert to proper size
        x = x.view(x.size(0), 1, 28, 28)

        # Initial convolution
        x = self.conv1(x)
        
        # Residual block 1 with skip connection
        residual = x
        x = self.res_block1(x)
        x += residual  # Skip connection
        x = F.relu(x)  # Activation after skip
        
        # Downsampling and residual block 2
        x = self.downsample1(x)
        residual = x
        x = self.res_block2(x)
        x += residual
        x = F.relu(x)
        
        # Downsampling and residual block 3
        x = self.downsample2(x)
        residual = x
        x = self.res_block3(x)
        x += residual
        x = F.relu(x)
        
        # Flatten for fully connected layers
        x = x.view(x.size(0), -1)
        
        # Fully connected layers
        x = self.fc1(x)
        x = self.fc2(x)
        x = self.fc3(x)
        
        return x

In [2]:
import mnist

BATCH_SIZE = 128
training_loader, test_loader = mnist.get_loaders(BATCH_SIZE)

In [3]:
model = UltraAdvancedCNN()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.0001)
EPOCHS = 30

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=0.001,
    steps_per_epoch=len(training_loader),
    epochs=EPOCHS,
    pct_start=0.3,
    div_factor=25.0,
    final_div_factor=1e4
)

In [ ]:
import trainer

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.mps.is_available()
    else "cpu"
)

trainer.train_model(
    model,
    training_loader,
    test_loader,
    loss_fn,
    optimizer,
    scheduler,
    print_freq=600,
    epochs=EPOCHS,
    device=device
)

In [5]:
# save the model
ts_model = torch.jit.script(model)
torch.jit.save(ts_model, "torchscript-models/mega-model.pt")